# Demo 6 — Ascoltare un impulso che diventa un suono risonante

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-3485c94980f2f6df")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        url = str(location.origin) + prefisso + "/_jupyterlite/files/audiolab-course.zip?v=3485c94980f2f6df"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 44100
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
SR_R, f0 = 10_000, 500
w0 = 2*np.pi*f0/SR_R
imp_r = np.zeros(SR_R)
imp_r[0] = 1.
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Oscillazione smorzata", "Picco di risonanza"])
versioni_r = []
for R in (0.95, 0.99, 0.995):
    G = (1-R)*np.sqrt(1-2*R*np.cos(2*w0)+R**2)
    a_r = [1., -2*R*np.cos(w0), R**2]
    h_r = al.filter_signal([G], a_r, imp_r)
    f_r, H_r = signal.freqz([G], a_r, worN=32768, fs=SR_R)
    fig.add_trace(go.Scatter(x=np.arange(1000)/SR_R, y=h_r[:1000],
                            name=f"R = {R}"), row=1, col=1)
    fig.add_trace(go.Scatter(x=f_r, y=al.to_db(np.abs(H_r), floor=1e-6), name=f"R = {R}",
                            showlegend=False), row=1, col=2)
    versioni_r.append((f"Impulso filtrato, R = {R}", h_r))
    # Misura sulla griglia: approssimazione della larghezza a -3 dB.
    k = np.argmax(np.abs(H_r))
    sopra = np.flatnonzero(np.abs(H_r) >= np.abs(H_r[k])/np.sqrt(2))
    B_mis = f_r[sopra[-1]]-f_r[sopra[0]]
    print(f"R={R}: picco {f_r[k]:.2f} Hz; B misurata {B_mis:.2f} Hz; "
          f"B approssimata {SR_R*(1-R)/np.pi:.2f} Hz")
fig.update_xaxes(title_text="Tempo [s]", row=1, col=1)
fig.update_yaxes(title_text="Ampiezza", row=1, col=1)
fig.update_xaxes(title_text="Frequenza [Hz]", range=[300, 700], row=1, col=2)
fig.update_yaxes(title_text="Guadagno [dB]", range=[-40, 3], row=1, col=2)
fig.update_layout(height=390)
fig.show()
# Guadagno comune esplicito per rendere udibili risposte impulsive deboli.
print("Ascolti: guadagno comune ×8, seguito dall'eventuale attenuazione comune.")
al.compare_audio([(nome, 8*h) for nome, h in versioni_r], SR_R)
